TF-IDF scores each word in an error message by how useful it is for telling categories apart. A word that shows up in almost every error (like "error" or "line") gets a low score, since it doesn't help distinguish one category from another. A word that's rare but specific to certain errors (like "subscriptable" or "iterable") gets a high score, since seeing it strongly suggests a particular category. This works well here because our error messages are short and technical — a handful of distinctive words usually carry most of the signal.

In [2]:
import os

os.chdir(r"D:\DevMentor AI")
print("Current working directory:", os.getcwd())
print("Does ml_engine exist here?", os.path.exists("ml_engine"))
print("Does ml_engine/data/labeled exist here?", os.path.exists("ml_engine/data/labeled"))

Current working directory: D:\DevMentor AI
Does ml_engine exist here? True
Does ml_engine/data/labeled exist here? True


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
print("TF-IDF import successful")

TF-IDF import successful


In [4]:
from client.runner import get_output_error, parse_error
from sanitizer.sanitizer import sanitize
import pandas as pd

scripts_and_labels = [
    ("ml_engine/data/raw/Syntax_Error.py", "syntax_error"),
    ("ml_engine/data/raw/Type_Error.py", "type_error"),
    ("ml_engine/data/raw/None_Type_Error.py", "none_type_error"),
    ("ml_engine/data/raw/Key_Error.py", "key_error"),
    ("ml_engine/data/raw/Index_Error.py", "index_error"),
    ("ml_engine/data/raw/Attribute_Error.py", "attribute_error"),
    ("ml_engine/data/raw/Module_Not_Found_Error.py", "module_not_found"),
    ("ml_engine/data/raw/File_Not_Found_Error.py", "file_not_found"),
    ("ml_engine/data/raw/permission_Error.py", "permission_error"),
    ("ml_engine/data/raw/value_Error.py", "value_error"),
    ("ml_engine/data/raw/network_Error.py", "network_error"),
    ("ml_engine/data/raw/Recursion_Error.py", "other_error"),
]

rows = []
for script_path, label in scripts_and_labels:
    error = get_output_error(script_path)
    result = parse_error(error)
    if result:
        error_type, message = result
        sanitized_message = sanitize(message)
        rows.append({"text": sanitized_message, "label": label})

df = pd.DataFrame(rows)
df.to_csv("ml_engine/data/labeled/dataset.csv", index=False)
print(df["label"].value_counts())

label
syntax_error        1
type_error          1
none_type_error     1
key_error           1
index_error         1
attribute_error     1
module_not_found    1
file_not_found      1
permission_error    1
value_error         1
network_error       1
other_error         1
Name: count, dtype: int64


In [5]:
print(df.head(12))

                                                 text             label
0                                        expected ':'      syntax_error
1   unsupported operand type(s) for +: 'int' and '...        type_error
2              'NoneType' object is not subscriptable   none_type_error
3                                             'email'         key_error
4                             list index out of range       index_error
5            'BuyBook' object has no attribute 'loan'   attribute_error
6                       No module named 'prettytable'  module_not_found
7   [Errno 2] No such file or directory: 'non_exis...    file_not_found
8        [Errno 13] Permission denied: 'readonly.txt'  permission_error
9   invalid literal for int() with base 10: 'twent...       value_error
10  HTTPConnectionPool(host='127.0.0.1', port=1): ...     network_error
11                   maximum recursion depth exceeded       other_error


In [6]:
def make_row(code_snippet: str, label: str) -> dict:
    """Runs a code snippet, captures its error message, sanitizes it,
    and returns a dataset row. Returns None if the snippet didn't give an error."""
    import subprocess, sys
    result = subprocess.run(
        [sys.executable, "-c", code_snippet],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        return None  # snippet didn't actually gave an error — skip it

    from client.runner import parse_error
    from sanitizer.sanitizer import sanitize
    
    parsed = parse_error(result.stderr)
    if not parsed:
        return None
    
    error_type, message = parsed
    return {"text": sanitize(message), "label": label}

In [7]:
key_error_snippets = [
    'user = {"name": "apurva"}; print(user["email"])',
    'config = {"debug": True}; print(config["timeout"])',
    'data = {"id": 1, "name": "test"}; print(data["value"])',
    'settings = {"theme": "dark"}; print(settings["language"])',
    'response = {"status": 200}; print(response["body"])',
    'cache = {}; print(cache["missing_key"])',
    'headers = {"Content-Type": "json"}; print(headers["Authorization"])',
    'params = {"page": 1}; print(params["limit"])',
    'record = {"id": 5}; print(record["timestamp"])',
    'payload = {"user_id": 42}; print(payload["session_token"])',
    'options = {"verbose": False}; print(options["quiet"])',
    'metadata = {"version": "1.0"}; print(metadata["build"])',
]

new_rows = []
for snippet in key_error_snippets:
    row = make_row(snippet, "key_error")
    if row:
        new_rows.append(row)

print(f"Generated {len(new_rows)} key_error examples")
for r in new_rows:
    print(r)

Generated 12 key_error examples
{'text': "'email'", 'label': 'key_error'}
{'text': "'timeout'", 'label': 'key_error'}
{'text': "'value'", 'label': 'key_error'}
{'text': "'language'", 'label': 'key_error'}
{'text': "'body'", 'label': 'key_error'}
{'text': "'missing_key'", 'label': 'key_error'}
{'text': "'Authorization'", 'label': 'key_error'}
{'text': "'limit'", 'label': 'key_error'}
{'text': "'timestamp'", 'label': 'key_error'}
{'text': "'session_token'", 'label': 'key_error'}
{'text': "'quiet'", 'label': 'key_error'}
{'text': "'build'", 'label': 'key_error'}


In [8]:
index_error_snippets = [
    'items = [1, 2, 3]; print(items[10])',
    'names = ["a", "b"]; print(names[5])',
    'results = []; print(results[0])',
    'queue = [1]; print(queue[3])',
    'stack = ["x", "y", "z"]; print(stack[10])',
    'buffer = [0, 1]; print(buffer[-5])',
    'rows = [[1,2],[3,4]]; print(rows[5])',
    'history = []; print(history[-1])',
    'batch = [1,2,3,4,5]; print(batch[100])',
    'tokens = ["a"]; print(tokens[1])',
    'matrix = [[1]]; print(matrix[0][5])',
    'ids = [10,20,30]; print(ids[7])',
]

new_rows = []
for snippet in index_error_snippets:
    row = make_row(snippet, "index_error")
    if row:
        new_rows.append(row)

print(f"Generated {len(new_rows)} index_error examples")
for r in new_rows:
    print(r)

Generated 12 index_error examples
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}
{'text': 'list index out of range', 'label': 'index_error'}


In [9]:
type_error_snippets = [
    'x = 5 + "text"',
    'y = "hello" + 10',
    'z = [1,2] + "a"',
    'result = None + 5',
    'total = 5 * text',
    'val = len(5)',
    'combined = {} + {}',
    'output = 3 / "two"',
    'items = [1,2,3]; items + 5',
    'name = "test"; name()',
    'count = 10; count["key"]',
    'value = True + "yes"',
]

new_rows = []
for snippet in type_error_snippets:
    row = make_row(snippet, "type_error")
    if row:
        new_rows.append(row)

print(f"Generated {len(new_rows)} type_error examples")
for r in new_rows:
    print(r)

for snippet in type_error_snippets:
    row = make_row(snippet, "type_error")
    if row is None:
        print("Did not produce a row:", snippet)

Generated 12 type_error examples
{'text': "unsupported operand type(s) for +: 'int' and 'str'", 'label': 'type_error'}
{'text': 'can only concatenate str (not "int") to str', 'label': 'type_error'}
{'text': 'can only concatenate list (not "str") to list', 'label': 'type_error'}
{'text': "unsupported operand type(s) for +: 'NoneType' and 'int'", 'label': 'type_error'}
{'text': "name 'text' is not defined. Did you mean: 'next'?", 'label': 'type_error'}
{'text': "object of type 'int' has no len()", 'label': 'type_error'}
{'text': "unsupported operand type(s) for +: 'dict' and 'dict'", 'label': 'type_error'}
{'text': "unsupported operand type(s) for /: 'int' and 'str'", 'label': 'type_error'}
{'text': 'can only concatenate list (not "int") to list', 'label': 'type_error'}
{'text': "'str' object is not callable", 'label': 'type_error'}
{'text': "'int' object is not subscriptable", 'label': 'type_error'}
{'text': "unsupported operand type(s) for +: 'bool' and 'str'", 'label': 'type_error'}


In [10]:
none_type_error_snippets = [
    'x = None; print(x.upper())',
    'data = None; print(data["key"])',
    'result = None; print(len(result))',
    'user = None; print(user.name)',
    'response = None; print(response["status"])',
    'value = None; print(value + 5)',
    'obj = None; obj.method()',
    'items = None; print(items[0])',
    'config = None; print(config.get("key"))',
    'total = None; print(total * 2)',
    'record = None; print(record.id)',
    'output = None; print(output.append(1))',
]

new_rows = []
for snippet in none_type_error_snippets:
    row = make_row(snippet, "none_type_error")
    if row:
        new_rows.append(row)

print(f"Generated {len(new_rows)} none_type_error examples")
for r in new_rows:
    print(r)

Generated 12 none_type_error examples
{'text': "'NoneType' object has no attribute 'upper'", 'label': 'none_type_error'}
{'text': "'NoneType' object is not subscriptable", 'label': 'none_type_error'}
{'text': "object of type 'NoneType' has no len()", 'label': 'none_type_error'}
{'text': "'NoneType' object has no attribute 'name'", 'label': 'none_type_error'}
{'text': "'NoneType' object is not subscriptable", 'label': 'none_type_error'}
{'text': "unsupported operand type(s) for +: 'NoneType' and 'int'", 'label': 'none_type_error'}
{'text': "'NoneType' object has no attribute 'method'", 'label': 'none_type_error'}
{'text': "'NoneType' object is not subscriptable", 'label': 'none_type_error'}
{'text': "'NoneType' object has no attribute 'get'", 'label': 'none_type_error'}
{'text': "unsupported operand type(s) for *: 'NoneType' and 'int'", 'label': 'none_type_error'}
{'text': "'NoneType' object has no attribute 'id'", 'label': 'none_type_error'}
{'text': "'NoneType' object has no attribute

In [11]:
syntax_error_snippets = [
    'if True\n    print("test")',
    'def foo(:\n    pass',
    'for i in range(10)\n    print(i)',
    'x = (1 + 2',
    'print("hello"',
    'while True\n    break',
    'def bar():\nreturn 5',
    'class Foo\n    pass',
    'if x == 1:\nelse:\n    pass',
    'y = [1, 2, 3',
    'lambda x: x +',
    'try:\n    pass\nexcept\n    pass',
]

new_rows = []
for snippet in syntax_error_snippets:
    row = make_row(snippet, "syntax_error")
    if row:
        new_rows.append(row)

print(f"Generated {len(new_rows)} syntax_error examples")
for r in new_rows:
    print(r)

Generated 12 syntax_error examples
{'text': "expected ':'", 'label': 'syntax_error'}
{'text': 'invalid syntax', 'label': 'syntax_error'}
{'text': "expected ':'", 'label': 'syntax_error'}
{'text': "'(' was never closed", 'label': 'syntax_error'}
{'text': "'(' was never closed", 'label': 'syntax_error'}
{'text': "expected ':'", 'label': 'syntax_error'}
{'text': 'expected an indented block after function definition on line 1', 'label': 'syntax_error'}
{'text': "expected ':'", 'label': 'syntax_error'}
{'text': "expected an indented block after 'if' statement on line 1", 'label': 'syntax_error'}
{'text': "'[' was never closed", 'label': 'syntax_error'}
{'text': 'invalid syntax', 'label': 'syntax_error'}
{'text': "expected ':'", 'label': 'syntax_error'}


In [12]:
attribute_error_snippets = [
    'x = 5; x.append(1)',
    'name = "test"; name.push("x")',
    'items = [1,2,3]; items.get("key")',
    'num = 10; num.upper()',
    'data = {"a": 1}; data.append(2)',
    'value = 3.14; value.split(",")',
    'flag = True; flag.lower()',
    'text = "hello"; text.add("x")',
    'lst = [1,2]; lst.keys()',
    'n = 5; n.strip()',
    'obj = object(); obj.missing_method()',
    'tup = (1,2); tup.append(3)',
]

new_rows = []
for snippet in attribute_error_snippets:
    row = make_row(snippet, "attribute_error")
    if row:
        new_rows.append(row)

print(f"Generated {len(new_rows)} attribute_error examples")
for r in new_rows:
    print(r)

Generated 12 attribute_error examples
{'text': "'int' object has no attribute 'append'", 'label': 'attribute_error'}
{'text': "'str' object has no attribute 'push'", 'label': 'attribute_error'}
{'text': "'list' object has no attribute 'get'", 'label': 'attribute_error'}
{'text': "'int' object has no attribute 'upper'", 'label': 'attribute_error'}
{'text': "'dict' object has no attribute 'append'", 'label': 'attribute_error'}
{'text': "'float' object has no attribute 'split'", 'label': 'attribute_error'}
{'text': "'bool' object has no attribute 'lower'", 'label': 'attribute_error'}
{'text': "'str' object has no attribute 'add'", 'label': 'attribute_error'}
{'text': "'list' object has no attribute 'keys'", 'label': 'attribute_error'}
{'text': "'int' object has no attribute 'strip'", 'label': 'attribute_error'}
{'text': "'object' object has no attribute 'missing_method'", 'label': 'attribute_error'}
{'text': "'tuple' object has no attribute 'append'", 'label': 'attribute_error'}


In [13]:
def generate_rows(snippets, label):
    rows = []
    for s in snippets:
        row = make_row(s, label)
        if row:
            rows.append(row)
    return rows

key_error_rows = generate_rows(key_error_snippets, "key_error")
index_error_rows = generate_rows(index_error_snippets, "index_error")
type_error_rows = generate_rows(type_error_snippets, "type_error")
none_type_error_rows = generate_rows(none_type_error_snippets, "none_type_error")
syntax_error_rows = generate_rows(syntax_error_snippets, "syntax_error")
attribute_error_rows = generate_rows(attribute_error_snippets, "attribute_error")

print(len(key_error_rows), len(index_error_rows), len(type_error_rows),
      len(none_type_error_rows), len(syntax_error_rows), len(attribute_error_rows))

12 12 12 12 12 12


In [14]:
import pandas as pd

# Load the existing 12-row dataset 
existing_df = pd.read_csv("ml_engine/data/labeled/dataset.csv")

# Combine all the new rows generated today across all 6 categories
all_new_rows = (
    key_error_rows + index_error_rows + type_error_rows +
    none_type_error_rows + syntax_error_rows + attribute_error_rows
)

new_df = pd.DataFrame(all_new_rows)
combined_df = pd.concat([existing_df, new_df], ignore_index=True)

combined_df.to_csv("ml_engine/data/labeled/dataset.csv", index=False)
print(combined_df["label"].value_counts())

label
syntax_error        13
type_error          13
none_type_error     13
key_error           13
index_error         13
attribute_error     13
module_not_found     1
file_not_found       1
permission_error     1
value_error          1
network_error        1
other_error          1
Name: count, dtype: int64


In [15]:
module_not_found_snippets = [
    'import nonexistent_module',
    'import fake_package_xyz',
    'from missing_lib import something',
    'import totally_fake_module_123',
    'import unicorn_package',
    'from nowhere_module import func',
    'import banana_framework',
    'import xyzpackage',
    'from fake_lib.utils import helper',
    'import made_up_module',
    'import definitely_not_real',
    'from ghost_package import Thing',
]

module_not_found_rows = generate_rows(module_not_found_snippets, "module_not_found")
print(f"Generated {len(module_not_found_rows)} module_not_found examples")
for r in module_not_found_rows:
    print(r)

Generated 12 module_not_found examples
{'text': "No module named 'nonexistent_module'", 'label': 'module_not_found'}
{'text': "No module named 'fake_package_xyz'", 'label': 'module_not_found'}
{'text': "No module named 'missing_lib'", 'label': 'module_not_found'}
{'text': "No module named '[REDACTED_TOKEN]'", 'label': 'module_not_found'}
{'text': "No module named 'unicorn_package'", 'label': 'module_not_found'}
{'text': "No module named 'nowhere_module'", 'label': 'module_not_found'}
{'text': "No module named 'banana_framework'", 'label': 'module_not_found'}
{'text': "No module named 'xyzpackage'", 'label': 'module_not_found'}
{'text': "No module named 'fake_lib'", 'label': 'module_not_found'}
{'text': "No module named 'made_up_module'", 'label': 'module_not_found'}
{'text': "No module named 'definitely_not_real'", 'label': 'module_not_found'}
{'text': "No module named 'ghost_package'", 'label': 'module_not_found'}


In [16]:
file_not_found_snippets = [
    'open("missing_file.txt")',
    'open("does_not_exist.csv")',
    'open("fake_config.json")',
    'open("/nonexistent/path/file.txt")',
    'open("no_such_report.pdf")',
    'open("ghost_data.txt")',
    'open("nowhere/file.log")',
    'open("fake_settings.yaml")',
    'open("missing_image.png")',
    'open("no_file_here.xml")',
    'open("absent_data.db")',
    'open("phantom_script.py")',
]

file_not_found_rows = generate_rows(file_not_found_snippets, "file_not_found")
print(f"Generated {len(file_not_found_rows)} file_not_found examples")
for r in file_not_found_rows:
    print(r)

Generated 12 file_not_found examples
{'text': "[Errno 2] No such file or directory: 'missing_file.txt'", 'label': 'file_not_found'}
{'text': "[Errno 2] No such file or directory: 'does_not_exist.csv'", 'label': 'file_not_found'}
{'text': "[Errno 2] No such file or directory: 'fake_config.json'", 'label': 'file_not_found'}
{'text': "[Errno 2] No such file or directory: '[REDACTED_PATH]'", 'label': 'file_not_found'}
{'text': "[Errno 2] No such file or directory: 'no_such_report.pdf'", 'label': 'file_not_found'}
{'text': "[Errno 2] No such file or directory: 'ghost_data.txt'", 'label': 'file_not_found'}
{'text': "[Errno 2] No such file or directory: 'nowhere/file.log'", 'label': 'file_not_found'}
{'text': "[Errno 2] No such file or directory: 'fake_settings.yaml'", 'label': 'file_not_found'}
{'text': "[Errno 2] No such file or directory: 'missing_image.png'", 'label': 'file_not_found'}
{'text': "[Errno 2] No such file or directory: 'no_file_here.xml'", 'label': 'file_not_found'}
{'text': 

In [17]:
value_error_snippets = [
    'int("abc")',
    'int("twelve")',
    'float("not_a_number")',
    'int("3.14.15")',
    'int("")',
    'float("nan_value")',
    'int("12a")',
    'list("a,b").index("z")',
    'int("  ")',
    'float("1,000")',
    'int("0x1F")',
    'bool(maybe)', 
]

value_error_rows = generate_rows(value_error_snippets, "value_error")
print(f"Generated {len(value_error_rows)} value_error examples")
for r in value_error_rows:
    print(r)

for snippet in value_error_snippets:
    row = make_row(snippet, "value_error")
    if row is None:
        print("Did not produce a error:", snippet)

Generated 12 value_error examples
{'text': "invalid literal for int() with base 10: 'abc'", 'label': 'value_error'}
{'text': "invalid literal for int() with base 10: 'twelve'", 'label': 'value_error'}
{'text': "could not convert string to float: 'not_a_number'", 'label': 'value_error'}
{'text': "invalid literal for int() with base 10: '3.14.15'", 'label': 'value_error'}
{'text': "invalid literal for int() with base 10: ''", 'label': 'value_error'}
{'text': "could not convert string to float: 'nan_value'", 'label': 'value_error'}
{'text': "invalid literal for int() with base 10: '12a'", 'label': 'value_error'}
{'text': "'z' is not in list", 'label': 'value_error'}
{'text': "invalid literal for int() with base 10: '  '", 'label': 'value_error'}
{'text': "could not convert string to float: '1,000'", 'label': 'value_error'}
{'text': "invalid literal for int() with base 10: '0x1F'", 'label': 'value_error'}
{'text': "name 'maybe' is not defined", 'label': 'value_error'}


In [18]:
import pandas as pd

df = pd.read_csv("ml_engine/data/labeled/dataset.csv")
print(df.shape)
print(df["label"].value_counts())

# See exactly which rows are duplicated
duplicates = df[df.duplicated(subset=["text", "label"], keep=False)]
print(duplicates.sort_values("label"))

(84, 2)
label
syntax_error        13
type_error          13
none_type_error     13
key_error           13
index_error         13
attribute_error     13
module_not_found     1
file_not_found       1
permission_error     1
value_error          1
network_error        1
other_error          1
Name: count, dtype: int64
                                                 text            label
33                            list index out of range      index_error
35                            list index out of range      index_error
34                            list index out of range      index_error
32                            list index out of range      index_error
31                            list index out of range      index_error
30                            list index out of range      index_error
28                            list index out of range      index_error
27                            list index out of range      index_error
29                            list index out 

In [19]:
# Keep at most 3 copies of any exact duplicate (text + label combination)
df_trimmed = df.groupby(["text", "label"]).head(3).reset_index(drop=True)

print(df_trimmed.shape)
print(df_trimmed["label"].value_counts())

(70, 2)
label
type_error          13
key_error           13
attribute_error     13
none_type_error     12
syntax_error        10
index_error          3
module_not_found     1
file_not_found       1
permission_error     1
value_error          1
network_error        1
other_error          1
Name: count, dtype: int64


In [20]:
df_trimmed.to_csv("ml_engine/data/labeled/dataset.csv", index=False)
print("Saved:", df_trimmed.shape)

Saved: (70, 2)
